# Model Inference Demonstration
Run the cell below to load the model and test it with a few sample alerts.

In [ ]:
import os
import torch
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def load_label_map(csv_path='data/action_space.csv'):
    df = pd.read_csv(csv_path)
    return dict(zip(df['Index'], df['Action Name']))

def predict(text, model, tokenizer, label_map, device='cpu'):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        probs = torch.nn.functional.softmax(model(**inputs).logits, dim=-1)[0]
    
    top_probs, top_indices = torch.topk(probs, 3)
    print(f'Input Text: {text}')
    print('-' * 50)
    for i in range(3):
        prob = top_probs[i].item()
        label = label_map.get(top_indices[i].item(), 'Unknown')
        print(f'{i+1}. {label} ({prob:.2%})')
    print('=' * 50 + '\n')

model_path = 'experiments/EXP_20260713_001/checkpoints/best_model'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if os.path.exists(model_path):
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model.eval()
    label_map = load_label_map('data/action_space.csv')
    print(f'Model loaded successfully on {device}!\n')
    
    samples = [
        'Multiple failed login attempts detected from IP 192.168.1.55 targeting the admin account via SSH.',
        'An unauthorized process mimikatz.exe was blocked from executing on workstation-04 by the EDR.',
    ]
    
    for s in samples:
        predict(s, model, tokenizer, label_map, device)
else:
    print(f'Error: Model not found at {model_path}')
